# Week 4 - Day 5 - Machine Learning Exercises

**Course:** Developers Institute - Data / ML Bootcamp  
**Author:** Alex Goldbaum

Theoretical exercises on ML problem framing, feature selection, model evaluation
strategies, and ML design for diverse scenarios — supported by empirical analysis
on the Kaggle Loan Prediction dataset.


## Setup

If running on Google Colab, upload `loan_train.csv` via the Files panel before running
the code cells. If running locally, place the CSV in the same folder as this notebook.


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

CSV_PATH = 'loan_train.csv'
if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
    except Exception:
        raise FileNotFoundError('Place loan_train.csv next to this notebook or upload it in Colab.')

df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
df.head()


## Exercise 1: Loan Default Prediction — Problem and Data

### Problem statement
Build a **binary classification** model that estimates the probability that a loan
applicant will **default** (fail to repay) within a defined horizon (e.g., 12 months
after origination).

The model will support the credit-risk team by:
- Making approve/reject decisions more consistent.
- Calibrating interest rates to individual risk.
- Reducing delinquency rates and expected losses.

**Model output:** probability `P(default)` in `[0, 1]`, plus a binary decision with
a threshold tuned to the asymmetric cost of false positives vs false negatives.

### Required data
1. **Personal / demographic:** age, marital status, dependents, education, housing status.
2. **Financial:** income, expenses, DTI ratio, employment tenure, employment type, assets.
3. **Credit:** credit score (FICO / Equifax / Dicom), credit history, prior delinquencies, recent inquiries, credit utilization.
4. **Loan:** amount, term, rate, purpose, collateral, co-signers.
5. **Behavioral:** internal payment history, account usage patterns.

### Data sources
- Internal core banking system / CRM.
- Credit bureaus (Equifax, Experian, TransUnion, Dicom in Chile).
- External scoring providers.
- Public data (regional unemployment, macroeconomic indicators).
- Government verification (SII, IRS, AFIP).
- Open banking / PSD2 with explicit user consent.

### Key considerations
- **Privacy:** GDPR, Chilean law 19.628, etc.
- **Fairness:** avoid discrimination by gender, race, age.
- **Class imbalance:** defaults are typically <10% of the population → SMOTE, `class_weight`, threshold tuning.


## Exercise 2: Feature Selection (Kaggle Loan Prediction)

Below we explore the actual dataset, examine missing values, study how each
variable relates to `Loan_Status`, and quantify feature importance with a Random
Forest.


In [ ]:
# Quick EDA
print('Class balance:')
print(df['Loan_Status'].value_counts(normalize=True).round(3))
print('\nMissing values per column:')
print(df.isna().sum())


In [ ]:
# Approval rate by Credit_History (this is the strongest single signal)
ct = pd.crosstab(df['Credit_History'], df['Loan_Status'], normalize='index').round(3)
print('Approval rate by Credit_History:')
print(ct)


In [ ]:
# Approval rate by Property_Area and Education
for col in ['Property_Area', 'Education', 'Married', 'Self_Employed', 'Gender']:
    ct = pd.crosstab(df[col], df['Loan_Status'], normalize='index').round(3)
    print(f'\nApproval rate by {col}:')
    print(ct)


In [ ]:
# Preprocessing for modeling: impute, encode, engineer features
data = df.copy()
data['Credit_History'] = data['Credit_History'].fillna(data['Credit_History'].mode()[0])
data['LoanAmount']     = data['LoanAmount'].fillna(data['LoanAmount'].median())
data['Loan_Amount_Term'] = data['Loan_Amount_Term'].fillna(data['Loan_Amount_Term'].mode()[0])
for c in ['Gender', 'Married', 'Dependents', 'Self_Employed']:
    data[c] = data[c].fillna(data[c].mode()[0])

# Feature engineering
data['TotalIncome']     = data['ApplicantIncome'] + data['CoapplicantIncome']
data['LoanAmount_log']  = np.log1p(data['LoanAmount'])
data['DTI']             = data['LoanAmount'] / (data['TotalIncome'] * data['Loan_Amount_Term'] / 12).replace(0, np.nan)
data['DTI']             = data['DTI'].fillna(data['DTI'].median())

# Encode categoricals
for col in ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']:
    data[col] = LabelEncoder().fit_transform(data[col].astype(str))
data['Loan_Status'] = (data['Loan_Status'] == 'Y').astype(int)

drop_cols = ['Loan_ID']
X = data.drop(columns=drop_cols + ['Loan_Status'])
y = data['Loan_Status']
X.head()


In [ ]:
# Feature importance from a quick Random Forest
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Random Forest feature importance', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('\nRanked feature importance:')
print(importances.sort_values(ascending=False).round(4))


### Findings & justification
- **`Credit_History`** is by far the strongest predictor — applicants with a clean
  credit history are approved at a much higher rate. Keep it.
- **`TotalIncome` / `ApplicantIncome` / `DTI`** capture repayment capacity and are
  consistently in the top-5 by importance.
- **`LoanAmount` (log-transformed)** is informative — larger loans = greater exposure.
- **`Property_Area`** carries moderate signal (Semiurban applicants get approved more).
- **`Education`, `Married`, `Dependents`, `Self_Employed`** add marginal lift but are useful
  for stability estimation.
- **Drop `Loan_ID`** (pure identifier, no signal).
- **`Gender`** has weak signal in this dataset and introduces fairness and legal risk
  in production — recommended to exclude or audit carefully.


## Exercise 3: Model Choice, Training and Evaluation

### Recommended models
1. **Logistic Regression** — interpretable baseline; coefficients translate to odds;
   meets banking regulatory requirements for explainability.
2. **Gradient Boosting (XGBoost / LightGBM / CatBoost)** — best performance on tabular
   data; handles non-linear interactions with minimal preprocessing.
3. **Calibrated score + business rules** — combine the model with hard rules (e.g.,
   automatic rejection if `Credit_History = 0`).

### Evaluation pipeline
1. Stratified train/test split (80/20).
2. Stratified k-fold CV (k=5) on the training set for hyperparameter tuning.
3. Final hold-out evaluated **once** at the end.
4. Tune the decision threshold using a business cost curve (not necessarily 0.5).

### Relevant metrics (the dataset is imbalanced — accuracy alone is misleading)
- **ROC-AUC** — robust to imbalance.
- **PR-AUC** — more informative with strong imbalance.
- **Recall** on the positive class — how many real defaults we catch.
- **Precision** — of those rejected, how many were genuinely risky.
- **F1 / F-beta** — `beta>1` if recall matters more.
- **Confusion matrix** and **expected business cost**.
- **Calibration plot** — predicted probabilities should match observed frequencies.

Below we train Logistic Regression and Random Forest on this dataset and compare them.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    cv_auc = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc').mean()
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    test_auc = roc_auc_score(y_test, proba)
    results[name] = {'cv_auc': cv_auc, 'test_auc': test_auc, 'proba': proba, 'preds': preds}
    print(f'{name}:')
    print(f'  CV ROC-AUC (5-fold): {cv_auc:.4f}')
    print(f'  Test ROC-AUC:        {test_auc:.4f}')
    print(classification_report(y_test, preds, target_names=['Default (N)', 'Repaid (Y)']))


In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred N', 'Pred Y'], yticklabels=['True N', 'True Y'])
    ax.set_title(f'{name}\nTest ROC-AUC: {res["test_auc"]:.3f}')
plt.tight_layout()
plt.show()


In [ ]:
# ROC curves
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {res["test_auc"]:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Loan Prediction', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


### Optimization roadmap
- Grid / Random / Bayesian search over hyperparameters.
- `class_weight='balanced'` or SMOTE for the class imbalance.
- Iterative feature engineering guided by SHAP / permutation importance.
- Production monitoring: feature drift and AUC degradation over time.


## Exercise 4: ML Type for Each Scenario

### 1) Predicting stock prices
**Type:** Supervised Learning — regression (time series).

**Why:** continuous target (future price); historical data is labeled with the
actual observed price at `t+1`, `t+5`, etc.

**Candidate models:** ARIMA / SARIMA (baseline), Gradient Boosting with technical
features (RSI, MACD, moving averages), LSTM or Transformers for long sequences.

**Note:** markets are noisy and near-efficient — marginal improvements over a
baseline are the realistic goal; beware of overfitting.

---

### 2) Organizing a library by genres
**Type:** Unsupervised Learning — clustering.

**Why:** no prior genre labels (if we had them, this would be supervised
classification). The goal is to **discover** structure by grouping books by
content similarity.

**Candidate models:** K-Means or Agglomerative Clustering on embeddings
(TF-IDF or sentence-transformers), HDBSCAN for variable-sized clusters, LDA for
interpretable topics.

---

### 3) Robot navigating a maze
**Type:** Reinforcement Learning (or classical search).

**Why:** an agent takes sequential actions and receives rewards (reach the goal,
avoid walls). There is no supervised dataset of `(state, optimal_action)` pairs.

**Candidate models:** Q-Learning / SARSA for discrete spaces, DQN if the state is
high-dimensional (e.g., pixels).

**Note:** if the maze is known and static, **A\*** or **BFS** find the optimal
solution without learning. RL shines when the environment is stochastic, partially
observable, or changes over time.


## Exercise 5: Evaluation Strategy for Three Model Types

### A) Supervised — Classification (e.g., spam vs non-spam)
**Strategy:** stratified train/val/test split + k-fold CV (k=5 or 10).

**Metrics:** Accuracy (only if balanced), Precision / Recall, F1, ROC-AUC, PR-AUC,
confusion matrix.

**Challenges:** class imbalance, data leakage in CV, concept drift in production.

---

### B) Unsupervised — Clustering (e.g., customer K-Means)
**Strategy:** no ground truth → indirect evaluation.

**Internal metrics:** Silhouette score, Davies-Bouldin, Calinski-Harabasz, Elbow
method (WCSS vs k).

**External metrics (if partial labels exist):** Adjusted Rand Index, Normalized
Mutual Information.

**Qualitative:** human inspection of clusters + stability across random seeds.

**Challenges:** 'good' depends on the objective, no ground truth; K-Means assumes
spherical clusters; curse of dimensionality.

---

### C) Reinforcement Learning (e.g., game-playing agent)
**Strategy:** train in a simulated environment, measure across N random seeds over
full episodes.

**Metrics:** cumulative reward per episode (mean + std across seeds), convergence,
sample efficiency, exploration vs exploitation balance (epsilon / policy entropy),
success rate.

**Challenges:** sparse rewards, training instability across seeds, generalization
to environments outside the training distribution, high computational cost.


## Conclusion

We covered the full conceptual cycle of an ML project: framing the problem and
data needs, selecting features with technical and business rigor (including
fairness), choosing the right model family for the problem type (supervised,
unsupervised, reinforcement), and designing an evaluation strategy that combines
technical metrics with business considerations and production monitoring.

The empirical section on the Kaggle Loan Prediction dataset confirmed that
`Credit_History` is the dominant feature, and that Logistic Regression already
achieves a strong ROC-AUC, with Random Forest providing comparable performance
and richer non-linear interactions.
